## Model Selection and Parameter Optimization

In this notebook, we will demonstrate how the NVIDIA NeMo Agent toolkit (NAT) optimizer can be used to create a robust model evaluation, comparison, and selection pipeline for custom datasets.

**Goal**:

By the end of this notebook, you will be able to:
- Build an LLM-as-a-judge evaluation for a simple chat workflow: define evaluators and optimizer settings, create an eval dataset, run the optimizer, and interpret results.
- Select optimial numeric parameters for a react agent.
- Select optimial prompt for llm inside a react agent.
- Weigh trade-offs across accuracy, groundedness, relevance, latency, and token efficiency, and export an optimized config for downstream production use.

- [0.0) Setup](#setup)
  - [0.1) Prerequisites](#prereqs)
  - [0.2) API Keys](#api-keys)
  - [0.3) Installing NeMo Agent Toolkit](#install-nat)
- [1.0) LLM-as-a-judge with NAT](#llm-judge-h1)
  - [1.1) LLM-as-a-judge workflow config](#config)
  - [1.2) Create an eval dataset](#dataset)
- [2.0) Hyperparameter Optimization - Numeric Parameters](#numeric)
- [3.0) Hyperparameter Optimization - Prompt](#prompt)
- [4.0) Next steps](#next-steps)

<a id="setup"></a>
# 0.0) Setup

<a id="prereqs"></a>
## 0.1) Prerequisites

We strongly recommend that users begin this notebook with a working understanding of NAT workflows. Please refer to earlier iterations of this notebook series prior to beginning this notebook.

- **Platform:** Linux, macOS, or Windows
- **Python:** version 3.11, 3.12, or 3.13
- **Python Packages:** `pip`

<a id="api-keys"></a>
## 0.2) API Keys

For this notebook, you will need the following API keys to run all examples end-to-end:

- **NVIDIA Build:** You can obtain an NVIDIA Build API Key by creating an [NVIDIA Build](https://build.nvidia.com) account and generating a key at https://build.nvidia.com/settings/api-keys

Then you can run the cell below:

In [ ]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

<a id="install-nat"></a>
## 0.3) Installing NeMo Agent Toolkit

The recommended way to install NAT is through `pip` or `uv pip`.

First, we will install `uv` which offers parallel downloads and faster dependency resolution.

In [ ]:
!pip install uv

NeMo Agent toolkit can be installed through the PyPI `nvidia-nat` package.

There are several optional subpackages available for NAT. For this example, we will rely on three subpackages:
* The `nvidia-nat[langchain]` subpackage contains components for integrating with [LangChain](https://python.langchain.com/docs/introduction/).
* The `nvidia-nat[profiling]` subpackage contains components for profiling and performance analysis.

In [ ]:
%%bash
uv pip show -q "nvidia-nat-langchain"
nat_langchain_installed=$?
uv pip show -q "nvidia-nat-profiling"
nat_profiling_installed=$?
if [[ ${nat_langchain_installed} -ne 0 || ${nat_profiling_installed} -ne 0 ]]; then
    cd ../../
    uv pip install -e ".[langchain,profiling]"
else
    echo "nvidia-nat[langchain,profiling] is already installed"
fi

<a id="llm-judge-h1"></a>
# 1.0) LLM-as-a-judge with NAT

The `nat eval` and `nat optimize` utilities enable developers to easily integrate LLM-as-a-judge capabilities with their workflows. `nat eval` allows for simple evaluations of a NAT workflow against an eval dataset. `nat optimize` extends this functionality by integrating with the **Optuna** library to perform grid and stochastic parameter sweeps and evaluations to identify optimal configurations for a task.

**Note:** _In this notebook, we will primarily demonstrate how to use `nat optimize` to identify a potentially optimal set of parameters for a NAT workflow. It is assumed that users will already have a strong understanding of ML model evaluations before building this concept into their workflows - as we will not be covering cross validation and train, validation, and test splitting of datasets. Please refer to python's [SciKit-Learn](https://scikit-learn.org/stable/) package as a strong reference for these concepts._

Let's look at the default configuration of this agent and confirm the agent type, LLMs, tool calls, and functions...

In [ ]:
%load getting_started/configs/config.yml

Now let's run this workflow for a simple Q&A example...

In [ ]:
!nat run --config_file getting_started/configs/config.yml --input "Suggest a single name for my new dog"

<a id="config"></a>
### 1.1) LLM-as-a-judge workflow config

In the cell below we edit our initial workflow configuration to include `eval` and `optimizer` configurations.

Key components of this configuration:

**LLM Configuration:**
- `nim_llm`: The backbone LLM that powers the workflow

**Judge LLM:**
- `nim_judge_llm`: A separate, more capable LLM used by the evaluator to assess the quality of the workflow's outputs
  - This LLM acts as an "LLM-as-a-judge" to score responses

**Evaluation Components:**
- `evaluators`: Define metrics to measure workflow quality (for example, accuracy, relevance)
- `profiler`: Instruments the workflow to collect performance metrics (latency, token usage, costs)


In [ ]:
%%writefile getting_started/configs/config_optimizer_numeric.yml
functions:
  current_datetime:
    _type: current_datetime

llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.0
    max_tokens: 1024
  
  nim_judge_llm:
    _type: nim
    model_name: qwen/qwen3-235b-a22b
    temperature: 0.0
    max_tokens: 8 # RAGAS accuracy only needs a score (0-1)
    thinking: false

workflow:
  _type: react_agent
  llm_name: nim_llm
  tool_names: [current_datetime]
  parse_agent_response_max_retries: 3

eval:
  general:
    output_dir: ./getting_started/eval_output
    verbose: true
    dataset:
      _type: json
      file_path: ./getting_started/eval_data.json
  
  evaluators:
    answer_accuracy:
      _type: ragas
      metric: AnswerAccuracy
      llm_name: nim_judge_llm
    llm_latency:
      _type: avg_llm_latency
    token_efficiency:
      _type: avg_tokens_per_llm_end


<a id="dataset"></a>
### 1.2) Create an eval dataset

The dataset below is intended to be difficult for simple LLM chat completions, because:
- Math calculations (questions 1, 2) require precise arithmetic that LLMs often struggle with
- Real-time data queries (questions 3) need current information beyond the model's training cutoff
- Factual knowledge (questions 4) may be outdated or incorrect without access to recent data
- Multi-step reasoning (questions 2) requires combining multiple operations accurately

In [ ]:
%%writefile getting_started/eval_data.json
[
    {
        "id": "1",
        "question": "What is 15% of 847?",
        "answer": "The answer is 127.05"
    },
    {
        "id": "2",
        "question": "If I invest $10,000 at 5% annual interest compounded monthly for 3 years, how much will I have?",
        "answer": "Approximately $11,614.72"
    },
    {
        "id": "3",
        "question": "What is the current weather in Tokyo?",
        "answer": "This requires real-time weather data for Tokyo, Japan."
    },
    {
        "id": "4",
        "question": "Who won the FIFA World Cup in 2022 and where was it held?",
        "answer": "Argentina won the 2022 FIFA World Cup, which was held in Qatar."
    },
    {
        "id": "5",
        "question": "Calculate the average of these numbers: 23, 45, 67, 89, 12, 34",
        "answer": "The average is 45"
    }
]

<a id="numeric"></a>
### 2.0) Hyperparameter Optimization - Numeric Parameters

**For a complete reference of all optimizer configuration parameters, see the [Optimizer documentation](../../docs/source/reference/optimizer.md) or go to your working branch on [GitHub - dev](https://github.com/NVIDIA/NeMo-Agent-Toolkit/blob/develop/docs/source/reference/optimizer.md).**

**Settings**

**LLM Configuration:**

- `optimizable_params`: Specifies which parameters the optimizer can tune (model name, temperature)
- `search_space`: Defines the values the optimizer will explore during optimization
  Must specify either:
  - Explicit values: `[0.5, 0.7, 0.9]`
  - Range with step: `low: 0.0, high: 1.0, step: 0.1`

**Optimizer Components:**
- `output_path`: Save all optimization results
- `reps_per_param_set`: Number of times to evaluate each parameter combination for statistical reliability
- `grid_search`: Strategy for exploring the search space (tests all combinations)
- `eval_metrics`: Metrics used to guide optimization decisions (for example, maximize accuracy while minimizing cost)
  - `evaluator_name`: References an evaluator you've defined in the `evaluators` (must match exactly)
  - `direction`:
    - `maximize` - Higher scores are better (accuracy, precision, F1)
    - `minimize` - Lower scores are better (latency, cost, error rate)
  - `weight`: coefficient of relative importance for the optimizer (defaults to 1.0)
- `numeric`: Controls how numeric (and categorical) parameters are optimized (uses Optuna library).
  - `enabled`: Set to `true` to turn on optimization of numeric parameters (like `temperature`, `max_tokens`, model selection)
  - `sampler`: Determines the search strategy for finding the best parameters 
    Options:
      - `grid`: Exhaustive search: tests every combination of parameter values 
      - `bayesian` or `null`: Smart search: uses Bayesian optimization to intelligently sample promising areas
- `prompt`: Controls genetic algorithm-based prompt optimization. 
  - `enabled`: Set to `false` when doing numeric optimization.



In [ ]:
%%writefile getting_started/configs/config_optimizer_numeric.yml
functions:
  current_datetime:
    _type: current_datetime

llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.0
    max_tokens: 1024
    optimizable_params:
      - model_name
      - temperature
    search_space:
      model_name:
        values:
          - meta/llama-3.1-70b-instruct
          - meta/llama-3.1-8b-instruct
      temperature:
        low: 0.0
        high: 0.4
        step: 0.2

  nim_judge_llm:
    _type: nim
    model_name: qwen/qwen3-next-80b-a3b-instruct
    temperature: 0.0
    max_tokens: 8 # RAGAS accuracy only needs a score (0-1)

workflow:
  _type: react_agent
  llm_name: nim_llm
  tool_names: [current_datetime]
  parse_agent_response_max_retries: 3

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO

eval:
  general:
    output_dir: ./getting_started/eval_output/optimize_numeric
    verbose: true
    dataset:
      _type: json
      file_path: ./getting_started/eval_data.json

  evaluators:
    answer_accuracy:
      _type: ragas
      metric: AnswerAccuracy
      llm_name: nim_judge_llm
    llm_latency:
      _type: avg_llm_latency
    token_efficiency:
      _type: avg_tokens_per_llm_end


optimizer:
  output_path: ./getting_started/eval_output/optimize_numeric/
  reps_per_param_set: 1 # Number of times to evaluate EACH config (for statistical significance)
  eval_metrics: # specifies which evaluatin metrics to optimize for
    accuracy: # custom name for the metric
      evaluator_name: answer_accuracy  # References the evaluator defined under the 'eval' section
      direction: maximize
      weight: 1.0 # coefficient of relative importance for the optimizer (defaults to 1.0)
    token_efficiency: # custom name for the metric
      evaluator_name: token_efficiency # References the evaluator defined under the 'eval' section
      direction: minimize
      weight: 1.0
    latency: # custom name for the metric
      evaluator_name: llm_latency # References the evaluator defined under the 'eval' section
      direction: minimize
      weight: 1.0

  numeric:
    enabled: true # enables numeric and categorical parameters to be optimized
    sampler: grid # uses Optuna GridSearch to determine the unique parameter sets to evaluate

  prompt:
    enabled: false  # Disable for pure model and hyperparameter comparison

**How does Optimizer Work**

Optimizer will:
- Test different parameter combinations (models, settings, etc.)
- Run each combination 1 times for reliability
- Measure 3 things: accuracy (↑), token efficiency (↓), latency (↓)
- Use grid search to test every combination systematically
- Skip prompt optimization (only testing model/parameter combinations)

Example workflow (if testing 2 models × 3 temperatures):
- Total unique configurations: 6
- Runs per config: 1
- Total workflow runs: 6
- Result: Best config balancing accuracy, cost, and speed

Output:
- One "best" configuration file
- Detailed comparison of all tested configurations
- Visualizations showing trade-offs between metrics

**Run the optimizer**

<div style="color: red; font-style: italic;">
<strong>Developer warning:</strong> Running the optimizer can take significant time (~30 minutes for search space of n=10 using NeMo endpoints) and  LLM inference tokens. Double check your config for unneeded search parameters or reduce the number of samples in the evaluation dataset to reduce cost.
</div>

In [ ]:
!nat optimize --config_file getting_started/configs/config_optimizer_numeric.yml

**Interpreting the Output**

When all the experiments are done, you'll see the below files in your `output_path`
- `optimized_config.yml`: The best configuration found
- `trials_dataframe_params.csv`: Detailed results from all trials
- `config_numeric_trial_{N}.yml`: Individual trial configurations
- `plots/`: Pareto front visualizations (if multiple metrics)
- `workflow_output.json`: Raw execution results from running the workflow on each test case.
  - `output_items`: Array of workflow execution results for each test case in the dataset.
    - `id`: Test case ID
    - `question`: User query
    - `answer`: Ground truth answer defined in the dataset
    - `generated_answer`: Generated by the agent workflow
    - `intermediate_steps`: Detailed execution trace
  The workflow output provides complete observability into each execution, enabling detailed analysis of agent behavior, performance profiling, and debugging.
- `answer_accuracy_output.json`
- `workflow_output.json`
- `llm_latency_output.json`
- `token_efficiency_output.json`

In [ ]:
from pathlib import Path

import pandas as pd

# Load the optimizer results
trials_df_path = Path("getting_started/eval_output/optimize_numeric/trials_dataframe_params.csv")

if trials_df_path.exists():
    trials_df = pd.read_csv(trials_df_path)

    print("Grid Search Optimization Results")
    print("=" * 80)
    print("\nTrials Summary:")
    print(trials_df.to_string(index=False))
    print("\n" + "=" * 80)

The results above show:
 
**Grid Search Optimization Summary:**
- The optimizer evaluated all combinations of models and temperatures defined in the search space
- Each configuration was tested multiple times (repetitions) to account for variability
- Three key metrics were tracked: accuracy, token efficiency (tokens used), and latency (response time)

**Key Insights:**
 - Different models show different trade-offs between accuracy, efficiency, and speed
- Temperature settings affect response variability and quality
- The "Best Configuration" represents the optimal balance based on the weighted combination of all metrics
 
**Interpreting Your Results:**
When you run this optimization, look for:
- Which model/temperature combination achieves the highest aggregated accuracy
- How token efficiency varies between models (lower is more efficient)
- Latency differences (lower is faster)
- The confidence intervals to understand result stability

The optimizer automatically selects the best configuration and saves it to `optimized_config.yml` for use in production.

<a id="numeric"></a>
### 3.0) Hyperparameter Optimization - Prompt

**For more information see the [Optimizer documentation](../../docs/source/reference/optimizer.md) or go to your working branch on [GitHub - dev](https://github.com/NVIDIA/NeMo-Agent-Toolkit/blob/develop/docs/source/reference/optimizer.md).**

NAT uses a Genetic Algorithm (GA) to automatically optimize prompts through evolutionary search. This is a sophisticated approach that treats prompts as "individuals" in a population that evolves over multiple generations to find better-performing variations. The genetic algorithm is inspired by natural evolution and uses LLMs themselves to intelligently mutate and recombine prompts. Instead of random mutations like traditional GAs, NAT leverages the reasoning capabilities of LLMs to make informed changes to prompts.

**Settings**

**Requirements**

- Prompt parameters marked with `OptimizableField(space=SearchSpace(is_prompt=True))`
- LLM functions for generating prompt variations


**Optimizer Components**

- `prompt`:
    - `enabled`: Enable GA-based prompt optimization (default: `false`)
    - `ga_population_size`: Population size - larger populations increase diversity but cost more per generation (default: `10`)
    - `ga_generations`: Number of generations to evolve prompts (default: `5`)
    - `ga_offspring_size`: Number of offspring per generation - if `null`, defaults to `ga_population_size - ga_elitism`*
    - `ga_crossover_rate`: Probability of recombination between two parents for each prompt parameter (default: `0.7`)*
    - `ga_mutation_rate`: Probability of mutating a child's prompt parameter using the LLM optimizer (default: `0.1`)*
    - `ga_elitism`: Number of elite individuals copied unchanged to the next generation (default: `1`)*
    - `ga_selection_method`: Parent selection scheme - `tournament` (default) or `roulette`*
    - `ga_tournament_size`: Tournament size when using tournament selection (default: `3`)*
    - `ga_parallel_evaluations`: Maximum number of concurrent evaluations (default: `8`)*
    - `ga_diversity_lambda`: Diversity penalty strength to discourage duplicate prompt sets - `0.0` disables it (default: `0.0`)
    - `prompt_population_init_function`: Function name used to mutate base prompts to seed the initial population and perform mutations. NAT includes a built-in `prompt_init` Function you can use.*
    - `prompt_recombination_function`: Optional function name used to recombine two parent prompts into a child prompt. NAT includes a built-in `prompt_recombiner` Function you can use.*

Modify ReAct agent's system_prompt to be a `OptimizableField`

`NeMo-Agent-Toolkit/src/nat/agent/react_agent/register.py`

**Before**
```python
system_prompt: str | None = Field(
    default=None,
    description="Provides the SYSTEM_PROMPT to use with the agent")  # defaults to SYSTEM_PROMPT in prompt.py
```

**After**
```python
system_prompt: str | None = OptimizableField(
    default=None,
    description="Provides the SYSTEM_PROMPT to use with the agent",  # defaults to SYSTEM_PROMPT in prompt.py
    space=SearchSpace(
        is_prompt=True,
        prompt=SYSTEM_PROMPT,
        prompt_purpose="Provides the SYSTEM_PROMPT to use with the agent",
    ))
```


In [ ]:
%%writefile getting_started/configs/config_optimizer_prompt.yml
functions:
  current_datetime:
    _type: current_datetime
  prompt_init:
    _type: prompt_init
    optimizer_llm: prompt_optimizer_llm
    system_objective: "ChatBot agent that can answer user questions directly or by calling tools"
  prompt_recombination:
    _type: prompt_recombiner
    optimizer_llm: prompt_optimizer_llm
    system_objective: "ChatBot agent that can answer user questions directly or by calling tools"

llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.0
    max_tokens: 1024
    optimizable_params:
      - model_name
      - temperature
    search_space:
      model_name:
        values:
          - meta/llama-3.1-70b-instruct
          - meta/llama-3.1-8b-instruct
      temperature:
        low: 0.0
        high: 0.4
        step: 0.2

  nim_judge_llm:
    _type: nim
    model_name: qwen/qwen3-next-80b-a3b-instruct
    temperature: 0.0
    max_tokens: 8 # RAGAS accuracy only needs a score (0-1)
  
  prompt_optimizer_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.5
    max_tokens: 2048

workflow:
  _type: react_agent
  llm_name: nim_llm
  tool_names: [current_datetime]
  parse_agent_response_max_retries: 3
  optimizable_params:
    - system_prompt
  search_space:
    system_prompt:
      is_prompt: true
      prompt_purpose: "Guide the agent to effectively answer user questions directly or by calling tools"
      prompt: |
        Answer the following questions as best you can. You may ask the human to use the following tools:

        {tools}

        You may respond in one of two formats.
        Use the following format exactly to ask the human to use a tool:

        Question: the input question you must answer
        Thought: you should always think about what to do
        Action: the action to take, should be one of [{tool_names}]
        Action Input: the input to the action (if there is no required input, include "Action Input: None")
        Observation: wait for the human to respond with the result from the tool, do not assume the response

        ... (this Thought/Action/Action Input/Observation can repeat N times. If you do not need to use a tool, or after asking the human to use any tools and waiting for the human to respond, you might know the final answer.)
        Use the following format once you have the final answer:

        Thought: I now know the final answer
        Final Answer: the final answer to the original input question

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO

eval:
  general:
    output_dir: ./getting_started/eval_output/optimize_prompt/
    verbose: true
    dataset:
      _type: json
      file_path: ./getting_started/eval_data.json

  evaluators:
    answer_accuracy:
      _type: ragas
      metric: AnswerAccuracy
      llm_name: nim_judge_llm
    llm_latency:
      _type: avg_llm_latency
    token_efficiency:
      _type: avg_tokens_per_llm_end


optimizer:
  output_path: ./getting_started/eval_output/optimize_prompt/
  reps_per_param_set: 1 # Number of times to evaluate EACH config (for statistical significance)
  eval_metrics: # specifies which evaluatin metrics to optimize for
    accuracy: # custom name for the metric
      evaluator_name: answer_accuracy  # References the evaluator defined under the 'eval' section
      direction: maximize
      weight: 1.0 # coefficient of relative importance for the optimizer (defaults to 1.0)
    token_efficiency: # custom name for the metric
      evaluator_name: token_efficiency # References the evaluator defined under the 'eval' section
      direction: minimize
      weight: 1.0
    latency: # custom name for the metric
      evaluator_name: llm_latency # References the evaluator defined under the 'eval' section
      direction: minimize
      weight: 1.0

  numeric:
    enabled: false # disable numeric and categorical parameters to be optimized
    sampler: grid # uses Optuna GridSearch to determine the unique parameter sets to evaluate

  prompt:
    enabled: true # enable for prompt optimization
    prompt_population_init_function: prompt_init
    prompt_recombination_function: prompt_recombination
    ga_generations: 3
    ga_population_size: 5


**Run the optimizer**
<div style="color: red; font-style: italic;">
<strong>Developer warning:</strong> Running the optimizer can consume a significant amount of LLM inference tokens.
</div>

In [ ]:
!nat optimize --config_file getting_started/configs/config_optimizer_prompt.yml

**Initial prompt:**

In [ ]:
Answer the following questions as best you can. You may ask the human to use the following tools:

{tools}

You may respond in one of two formats.
Use the following format exactly to ask the human to use a tool:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action (if there is no required input, include "Action Input: None")
Observation: wait for the human to respond with the result from the tool, do not assume the response

... (this Thought/Action/Action Input/Observation can repeat N times.
If you do not need to use a tool, or after asking the human to use any tools and waiting for the human to respond, 
you might know the final answer.)
Use the following format once you have the final answer:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

**Optimized prompt:**

In [ ]:
To answer the user's question, follow these steps:

1. Read the input question carefully and think about the best approach to find the answer.
2. If you need to use a tool to find the answer, ask the human to use one of the following tools: {tools}. 
   To do this, respond in the following format:

   **Tool Usage Format**

   1. Question: the input question you must answer
   2. Thought: analyze the question and determine the best course of action
   3. Action: select one of the available tools [{tool_names}] to assist in answering the question
   4. Action Input: provide the necessary input for the selected tool (or "None" if no input is required)
   5. Observation: wait for the human to respond with the result from the tool, do not assume the response

   **Repeat as necessary**: You may repeat steps 2-5 until you have sufficient information to answer the question.
3. If you do not need to use a tool or after using a tool and receiving the result, think about the final answer.
4. Respond with the final answer in the following format:

   **Final Answer Format**

   1. Thought: I now know the final answer
   2. Final Answer: the final answer to the original input question

**Important Guidelines**

* Always use the exact format specified above.
* Only use the provided tools when necessary, and follow the specified action and input guidelines.
* If you encounter any issues or uncertainties, clarify with the human before proceeding.
* Ensure your final answer is accurate and relevant to the original question.

**Examples:**

* If the input question is "What is the capital of France?", you might respond:
  Question: What is the capital of France?
  Thought: I know this one
  Final Answer: Paris
* If the input question is "What is the definition of artificial intelligence?", you might respond:
  Question: What is the definition of artificial intelligence?
  Thought: I need to look this up
  Action: use the dictionary tool
  Action Input: artificial intelligence
  Observation: wait for the human to respond with the definition
  Thought: I now know the final answer
  Final Answer: [insert definition from dictionary tool]

Remember to always follow the specified format and only use the tools provided.

**Key differences between the prompts:**
The genetic algorithm optimization process made several significant improvements to the prompt structure and content:
1. **Enhanced Structure**: The optimized prompt adds explicit sections for **Objective** and **Constraints**, providing clearer context and boundaries for the agent's task.
2. **Corner Case Handling**: Explicitly pointed out that tools are not necessary, and what to do if a tool is not needed.

These changes demonstrate how the optimization process evolved the prompt from a compact, functional instruction set to a more comprehensive, structured guide that provides clearer expectations and examples for the agent to follow.
<!-- path-check-skip-end -->

<a id="next-steps"></a>
# 4.0) Next steps

Continue learning how to fully utilize the NVIDIA NeMo Agent toolkit by exploring the other documentation and advanced agents in the `examples` directory.

**Excercise**

1. Try to improve `system_prompt` to reduce irrelevant intermediate steps.
2. Add relavant tools to better handle the data queries.
